# iNaturalist Species Co-occurrence Network — Data Collection

Crawls research-grade observations from iNaturalist for a given geographic area
(default: Italy) and saves raw observation data for later network construction.

**Network model:**
- Nodes = species (taxon_id)
- Edges = spatial co-occurrence (two species observed in the same grid cell)
- Edge weight = number of grid cells where co-occurrence happens

**Rate limits (iNaturalist guidelines):**
- ~1 request/second
- ~10k requests/day
- max 200 results per page
- max 10k results per query (use id_above pagination)

In [ ]:
# ── Repository root ───────────────────────────────────────────────────────────
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ('notebooks', 'legacy') else Path.cwd()
assert (ROOT / 'data').is_dir(), f'Repository root not found from {Path.cwd()}'

In [1]:
import requests
import pandas as pd
import json
import time
import os
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm

## Configuration

In [ ]:
API_BASE = "https://api.inaturalist.org/v1"

# Italy bounding box (approximate)
ITALY_BBOX = {
    "swlat": 35.5,   # Southern tip (Lampedusa)
    "swlng": 6.6,    # Western border
    "nelat": 47.1,   # Northern border (Alps)
    "nelng": 18.5,   # Eastern border
}


# Grid cell size for co-occurrence (in degrees)
# 0.1° ≈ 11 km at Italian latitudes
GRID_CELL_SIZE = 0.1

# Output files
OUTPUT_DIR = ROOT / "data" / "raw"
OBS_FILE = OUTPUT_DIR / "observations.csv.gz"
SPECIES_FILE = OUTPUT_DIR / "species.csv.gz"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"
STATS_FILE = OUTPUT_DIR / "crawl_stats.json"

# Request settings
PER_PAGE = 200          # max allowed by iNaturalist
DELAY_SECONDS = 1.1     
USER_AGENT = "SNA-UniPi-Project/1.0 (species co-occurrence network)"

## API Interaction

In [3]:
def fetch_observations(params: dict) -> dict:
    """
    Fetch observations from iNaturalist API v1.

    Returns the full API response as a dict.
    Handles rate limiting and retries.
    """
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
    }

    url = f"{API_BASE}/observations"

    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=30)

            if response.status_code == 429:
                wait_time = 60 * (attempt + 1)
                print(f"\n⚠️  Rate limited! Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"\n⚠️  Request error (attempt {attempt+1}/3): {e}")
            if attempt < 2:
                time.sleep(10 * (attempt + 1))
            else:
                raise

    return {}


def get_total_count(bbox: dict, quality_grade: str = "research",
                    year: int = None) -> int:
    """Get total number of observations matching the query."""
    params = {
        **bbox,
        "quality_grade": quality_grade,
        "per_page": 0,
        "geo": "true",
        "verifiable": "true",
    }
    if year:
        params["d1"] = f"{year}-01-01"
        params["d2"] = f"{year}-12-31"
    result = fetch_observations(params)
    return result.get("total_results", 0)

In [ ]:
def extract_observation(obs: dict) -> dict | None:
    """
    Extract relevant fields from a raw observation.

    Returns None if the observation lacks required data
    (species-level identification or coordinates).
    """
    taxon = obs.get("taxon")
    if not taxon:
        return None

    rank = taxon.get("rank", "")
    if rank not in ("species", "subspecies", "variety", "form"):
        return None

    location = obs.get("location")
    if not location:
        return None

    try:
        lat, lng = map(float, location.split(","))
    except (ValueError, AttributeError):
        return None

    grid_lat = round(lat // GRID_CELL_SIZE * GRID_CELL_SIZE, 4)
    grid_lng = round(lng // GRID_CELL_SIZE * GRID_CELL_SIZE, 4)
    grid_cell = f"{grid_lat},{grid_lng}"

    # Parse date and assign season
    observed_on = obs.get("observed_on", "")
    month = None
    season = "unknown"
    if observed_on:
        try:
            month = int(observed_on.split("-")[1])
            # Meteorological seasons 
            if month in (12, 1, 2):
                season = "winter"
            elif month in (3, 4, 5):
                season = "spring"
            elif month in (6, 7, 8):
                season = "summer"
            elif month in (9, 10, 11):
                season = "autumn"
        except (IndexError, ValueError):
            pass

    return {
        "obs_id": obs["id"],
        "taxon_id": taxon["id"],
        "species_name": taxon.get("name", ""),
        "common_name": (
            taxon.get("preferred_common_name") or
            taxon.get("english_common_name", "")
        ),
        "iconic_taxon": taxon.get("iconic_taxon_name", ""),
        "latitude": lat,
        "longitude": lng,
        "grid_cell": grid_cell,
        "observed_on": observed_on,
        "month": month,
        "season": season,
        "place_guess": obs.get("place_guess", ""),
        "positional_accuracy": obs.get("positional_accuracy"),
    }


def extract_species_info(obs: dict) -> dict:
    """Extract species-level metadata for node attributes."""
    taxon = obs.get("taxon", {})

    ancestors = taxon.get("ancestors", [])
    taxonomy = {}
    for anc in (ancestors or []):
        r = anc.get("rank", "")
        if r in ("kingdom", "phylum", "class", "order", "family", "genus"):
            taxonomy[r] = anc.get("name", "")

    return {
        "taxon_id": taxon.get("id"),
        "species_name": taxon.get("name", ""),
        "common_name": (
            taxon.get("preferred_common_name") or
            taxon.get("english_common_name", "")
        ),
        "rank": taxon.get("rank", ""),
        "iconic_taxon": taxon.get("iconic_taxon_name", ""),
        "observations_count": taxon.get("observations_count", 0),
        "threatened": taxon.get("threatened", False),
        **taxonomy,
    }

## Persistence

In [4]:
def save_checkpoint(observations: list, species_info: dict, last_id: int):
    """Save current progress to disk."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.DataFrame(observations)
    df.to_csv(OBS_FILE, index=False)

    species_df = pd.DataFrame(species_info.values())
    species_df.to_csv(SPECIES_FILE, index=False)

    checkpoint = {
        "last_id": last_id,
        "total_observations": len(observations),
        "unique_species": len(species_info),
        "timestamp": datetime.now().isoformat(),
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)

    print(f"  💾 Checkpoint: {len(observations):,} obs, "
          f"{len(species_info):,} species (last_id={last_id})")

## Crawl

In [ ]:
def crawl_observations(
    bbox: dict,
    resume_from_id: int = 0,
    max_observations: int = None,
    year: int = None,
) -> tuple[list, dict]:
    """
    Crawl all research-grade observations in the bounding box.

    Uses id_above pagination (recommended by iNaturalist for large result sets).
    Saves checkpoints every 50 pages (~10k observations).

    Args:
        bbox: bounding box dict with swlat, swlng, nelat, nelng
        resume_from_id: resume crawl from this observation ID
        max_observations: stop after this many observations
        year: filter observations to this year only

    Returns (observations list, species_info dict).
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    total = get_total_count(bbox, year=year)
    print(f"📊 Total research-grade observations in area: {total:,}")
    if year:
        print(f"📅 Filtered to year: {year}")

    if max_observations:
        total = min(total, max_observations)
        print(f"📌 Limiting to {max_observations:,} observations")

    time.sleep(DELAY_SECONDS)

    observations = []
    species_info = {}  # taxon_id -> species metadata
    last_id = resume_from_id
    fetched = 0
    page_count = 0

    if resume_from_id > 0 and os.path.exists(OBS_FILE):
        existing_df = pd.read_csv(OBS_FILE)
        observations = existing_df.to_dict("records")
        fetched = len(observations)
        print(f"🔄 Resuming from id > {resume_from_id} ({fetched:,} existing records)")

    pbar = tqdm(total=total, initial=fetched, desc="Crawling observations",
                unit="obs", ncols=100)

    while True:
        params = {
            **bbox,
            "quality_grade": "research",
            "order_by": "id",
            "order": "asc",
            "per_page": PER_PAGE,
            "id_above": last_id,
            "geo": "true",
            "verifiable": "true",
            "fields": (
                "id,taxon,location,observed_on,place_guess,"
                "positional_accuracy,quality_grade"
            ),
        }
        if year:
            params["d1"] = f"{year}-01-01"
            params["d2"] = f"{year}-12-31"

        result = fetch_observations(params)
        results = result.get("results", [])

        if not results:
            break

        for obs in results:
            record = extract_observation(obs)
            if record:
                observations.append(record)
                tid = record["taxon_id"]
                if tid not in species_info:
                    species_info[tid] = extract_species_info(obs)

        fetched += len(results)
        last_id = results[-1]["id"]
        page_count += 1
        pbar.update(len(results))

        if page_count % 50 == 0:
            save_checkpoint(observations, species_info, last_id)

        if max_observations and fetched >= max_observations:
            print(f"\n Reached limit of {max_observations:,} observations")
            break

        if len(results) < PER_PAGE:
            break

        time.sleep(DELAY_SECONDS)

    pbar.close()
    save_checkpoint(observations, species_info, last_id)

    return observations, species_info

## Data Summary

In [ ]:
def print_data_summary(observations: list, species_info: dict):
    """Print a summary of the crawled data."""
    df = pd.DataFrame(observations)

    print("\n" + "=" * 60)
    print("CRAWL SUMMARY")
    print("=" * 60)
    print(f"Total observations:       {len(df):,}")
    print(f"Unique species (nodes):   {df['taxon_id'].nunique():,}")
    print(f"Unique grid cells:        {df['grid_cell'].nunique():,}")

    print(f"\n Observations by iconic taxon:")
    iconic_counts = df["iconic_taxon"].value_counts()
    for taxon, count in iconic_counts.head(10).items():
        pct = count / len(df) * 100
        print(f"  {taxon:20s}  {count:>8,}  ({pct:.1f}%)")

    print(f"\n🔝 Most observed species:")
    top_species = (
        df.groupby(["taxon_id", "species_name"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )
    for _, row in top_species.iterrows():
        print(f"  {row['species_name']:40s}  {row['count']:>6,} obs")

    print(f"\n Grid cell coverage:")
    cells_per_species = df.groupby("taxon_id")["grid_cell"].nunique()
    print(f"  Avg cells per species:  {cells_per_species.mean():.1f}")
    print(f"  Max cells per species:  {cells_per_species.max()}")
    print(f"  Median cells/species:   {cells_per_species.median():.0f}")

    print(f"\n🔗 Network size estimate:")
    species_per_cell = df.groupby("grid_cell")["taxon_id"].nunique()
    estimated_edges = sum(n * (n - 1) // 2 for n in species_per_cell)
    print(f"  Avg species per cell:   {species_per_cell.mean():.1f}")
    print(f"  Max species per cell:   {species_per_cell.max()}")
    print(f"  Estimated edges (raw):  {estimated_edges:,}")

    if "season" in df.columns:
        print(f"\n Seasonal breakdown:")
        for season in ["winter", "spring", "summer", "autumn"]:
            s_df = df[df["season"] == season]
            if len(s_df) == 0:
                continue
            print(f"  {season.capitalize():8s}:  {len(s_df):>8,} obs, "
                  f"{s_df['taxon_id'].nunique():>6,} species, "
                  f"{s_df['grid_cell'].nunique():>5,} cells")

        species_seasons = df.groupby("taxon_id")["season"].apply(set)
        single_season = species_seasons[species_seasons.apply(len) == 1]
        all_season = species_seasons[species_seasons.apply(len) == 4]
        print(f"\n  Single-season species:   {len(single_season):,}")
        print(f"  All-season species:      {len(all_season):,}")

    stats = {
        "total_observations": len(df),
        "unique_species": int(df["taxon_id"].nunique()),
        "unique_grid_cells": int(df["grid_cell"].nunique()),
        "estimated_edges": int(estimated_edges),
        "iconic_taxa": iconic_counts.to_dict(),
        "timestamp": datetime.now().isoformat(),
    }
    with open(STATS_FILE, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"\n✅ Data saved to {OUTPUT_DIR}/")
    print(f"   - {OBS_FILE} ({os.path.getsize(OBS_FILE) / 1e6:.1f} MB)")
    print(f"   - {SPECIES_FILE}")

## Run the Crawl

Edit the options below, then run the cell.

In [ ]:
# --- Options ---
AREA      = "italy"   # "italy", "tuscany", or "marche"
YEAR      = 2025        # filter to this year (None = all years)
TEST_MODE = False       # True = fetch only 1000 obs (quick test)
RESUME    = False       # True = resume from last checkpoint
MAX_OBS   = None        # e.g. 50_000 or None for all
# ---------------

BBOX_MAP = {
    "italy":   ITALY_BBOX,
}

if TEST_MODE:
    bbox = ITALY_BBOX  
    max_obs = 1000
    print(f"TEST MODE: Fetching 1000 observations from Marche ({YEAR})")
else:
    bbox = BBOX_MAP[AREA]
    max_obs = MAX_OBS
    print(f" Area: {AREA.upper()}")

print(f"📅 Year: {YEAR}")
print(f"📍 Bounding box: SW({bbox['swlat']}, {bbox['swlng']}) "
      f"NE({bbox['nelat']}, {bbox['nelng']})")

resume_id = 0
if RESUME and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
    resume_id = checkpoint["last_id"]
    print(f"🔄 Resuming from checkpoint (last_id={resume_id})")

start_time = time.time()
observations, species_info = crawl_observations(
    bbox, resume_from_id=resume_id, max_observations=max_obs, year=YEAR
)
elapsed = time.time() - start_time

print(f"Crawling took {elapsed/60:.1f} minutes")

if observations:
    print_data_summary(observations, species_info)
else:
    print("❌ No observations collected!")

🌍 Area: ITALY
📅 Year: 2025
📍 Bounding box: SW(35.5, 6.6) NE(47.1, 18.5)
📊 Total research-grade observations in area: 1,094,851
📅 Filtered to year: 2025


Crawling observations:   1%|▎                            | 10000/1094851 [05:15<9:45:41, 30.87obs/s]

  💾 Checkpoint: 9,934 obs, 1,938 species (last_id=258634663)


Crawling observations:   2%|▌                            | 20000/1094851 [10:08<8:25:22, 35.45obs/s]

  💾 Checkpoint: 19,845 obs, 2,698 species (last_id=260273801)


...

  💾 Checkpoint: 1,073,009 obs, 22,133 species (last_id=336612058)


Crawling observations: 100%|██████████████████████████▉| 1090000/1094851 [9:16:01<02:12, 36.61obs/s]

  💾 Checkpoint: 1,082,957 obs, 22,210 species (last_id=343364761)


Crawling observations: 1095008obs [9:18:24, 32.68obs/s]                                             


  💾 Checkpoint: 1,087,953 obs, 22,232 species (last_id=350183384)
Crawling took 558.7 minutes

📋 CRAWL SUMMARY
Total observations:       1,087,953
Unique species (nodes):   22,232
Unique grid cells:        6,662

📊 Observations by iconic taxon:
  Plantae                425,818  (39.1%)
  Insecta                339,148  (31.2%)
  Aves                   152,108  (14.0%)
  Fungi                   29,958  (2.8%)
  Reptilia                28,887  (2.7%)
  Arachnida               22,783  (2.1%)
  Mollusca                21,368  (2.0%)
  Mammalia                18,346  (1.7%)
  Amphibia                17,546  (1.6%)
  Animalia                17,544  (1.6%)

🔝 Most observed species:
  Podarcis muralis                           6,866 obs
  Anas platyrhynchos                         4,305 obs
  Larus michahellis                          3,921 obs
  Podarcis siculus                           3,857 obs
  Turdus merula                              3,845 obs
  Ailanthus altissima                    

## Supplementary Crawl — Winter Extension

The original crawl covers only calendar year 2025, which means the meteorological winter is incomplete:
- Winter 2024-25 is missing December 2024
- Winter 2025-26 is missing January–February 2026

The cell below crawls **only the 3 missing months** (Dec 2024, Jan 2026, Feb 2026),
then merges with the existing `observations.csv` and reassigns seasons with a
**5-snapshot timeline**:

| Snapshot | Months | Notes |
|----------|--------|-------|
| `winter_0` | Dec 2024 + Jan–Feb 2025 | Complete winter |
| `spring` | Mar–May 2025 | Unchanged |
| `summer` | Jun–Aug 2025 | Unchanged |
| `autumn` | Sep–Nov 2025 | Unchanged |
| `winter_1` | Dec 2025 + Jan–Feb 2026 | Complete winter |

This gives 4 clean transitions and allows comparing Winter₀ vs Winter₁
for inter-annual stability analysis.

**⚠️ Requires the existing `data/observations.csv` from the original crawl (2025).**
Does NOT re-run the original crawl.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Supplementary crawl: Dec 2024, Jan 2026, Feb 2026
# Reuses fetch_observations / extract_observation / extract_species_info
# ──────────────────────────────────────────────────────────────

SUPP_WINDOWS = [
    {"label": "Dec 2024", "d1": "2024-12-01", "d2": "2024-12-31"},
    {"label": "Jan 2026", "d1": "2026-01-01", "d2": "2026-01-31"},
    {"label": "Feb 2026", "d1": "2026-02-01", "d2": "2026-02-28"},
]

SUPP_FILE = OUTPUT_DIR / "observations_supplementary.csv.gz"
EXT_FILE  = OUTPUT_DIR / "observations_extended.csv.gz"

bbox = ITALY_BBOX
supp_observations = []
supp_species = {}

for window in SUPP_WINDOWS:
    print(f"\n{'='*60}")
    print(f"📅 Crawling {window['label']}  ({window['d1']} → {window['d2']})")
    print(f"{'='*60}")

    # Count
    count_params = {**bbox, "quality_grade": "research", "per_page": 0,
                    "geo": "true", "verifiable": "true",
                    "d1": window["d1"], "d2": window["d2"]}
    total = fetch_observations(count_params).get("total_results", 0)
    print(f"  Estimated observations: {total:,}")
    time.sleep(DELAY_SECONDS)

    # Paginate with id_above
    last_id = 0
    window_obs = []
    pbar = tqdm(total=total, desc=f"  {window['label']}", unit=" obs")

    while True:
        params = {
            **bbox,
            "quality_grade": "research",
            "order_by": "id", "order": "asc",
            "per_page": PER_PAGE,
            "id_above": last_id,
            "d1": window["d1"], "d2": window["d2"],
            "geo": "true", "verifiable": "true",
            "fields": "id,taxon,location,observed_on,place_guess,positional_accuracy,quality_grade",
        }
        result = fetch_observations(params)
        results = result.get("results", [])
        if not results:
            break

        for obs_raw in results:
            record = extract_observation(obs_raw)
            if record:
                window_obs.append(record)
                tid = record["taxon_id"]
                if tid not in supp_species:
                    supp_species[tid] = extract_species_info(obs_raw)

        last_id = results[-1]["id"]
        pbar.update(len(results))

        if len(results) < PER_PAGE:
            break
        time.sleep(DELAY_SECONDS)

    pbar.close()
    supp_observations.extend(window_obs)
    print(f"  ✅ {len(window_obs):,} animal observations")

# Save supplementary CSV
supp_df = pd.DataFrame(supp_observations)
supp_df.to_csv(SUPP_FILE, index=False)
print(f"\n💾 Supplementary saved: {SUPP_FILE} ({len(supp_df):,} rows)")

# ──────────────────────────────────────────────────────────────
# Merge with existing observations.csv and reassign seasons
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"🔄 Merging with existing observations...")
print(f"{'='*60}")

main_df = pd.read_csv(OBS_FILE)
print(f"  Original: {len(main_df):,} rows")
print(f"  Supplementary: {len(supp_df):,} rows")

full_df = pd.concat([main_df, supp_df], ignore_index=True)
before = len(full_df)
full_df = full_df.drop_duplicates(subset=["obs_id"], keep="first")
dupes = before - len(full_df)
if dupes > 0:
    print(f"  ⚠️  Removed {dupes:,} duplicates")

# Reassign season using year + month → 5-snapshot labels
def assign_season_5snap(row):
    date_str = str(row.get("observed_on", ""))
    if not date_str or date_str == "nan":
        return "unknown"
    try:
        parts = date_str.split("-")
        year, month = int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return "unknown"

    if year == 2024 and month == 12:      return "winter_0"
    if year == 2025 and month in (1, 2):  return "winter_0"
    if year == 2025 and month in (3,4,5): return "spring"
    if year == 2025 and month in (6,7,8): return "summer"
    if year == 2025 and month in (9,10,11): return "autumn"
    if year == 2025 and month == 12:      return "winter_1"
    if year == 2026 and month in (1, 2):  return "winter_1"
    return "unknown"

full_df["season"] = full_df.apply(assign_season_5snap, axis=1)
n_unknown = (full_df["season"] == "unknown").sum()
if n_unknown > 0:
    print(f"  ⚠️  {n_unknown:,} observations outside timeline (dropped)")
full_df = full_df[full_df["season"] != "unknown"].copy()

# Filter: keep only animal observations
non_animal = {"Plantae", "Fungi", "Chromista", "Protozoa", ""}
full_df = full_df[~full_df["iconic_taxon"].isin(non_animal)].copy()

full_df.to_csv(EXT_FILE, index=False)
print(f"\n💾 Extended dataset saved: {EXT_FILE} ({len(full_df):,} rows)")

# ──────────────────────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"📋 EXTENDED DATASET — 5-Snapshot Timeline")
print(f"{'='*60}")
print(f"  Total observations:  {len(full_df):,}")
print(f"  Unique species:      {full_df['taxon_id'].nunique():,}")
print(f"  Unique grid cells:   {full_df['grid_cell'].nunique():,}")
print()

season_order = ["winter_0", "spring", "summer", "autumn", "winter_1"]
labels = {"winter_0": "Winter₀ (Dec24+Jan-Feb25)",
          "spring":   "Spring  (Mar-May 2025)",
          "summer":   "Summer  (Jun-Aug 2025)",
          "autumn":   "Autumn  (Sep-Nov 2025)",
          "winter_1": "Winter₁ (Dec25+Jan-Feb26)"}

print(f"  {'Season':<30s} {'Obs':>8s} {'Species':>8s} {'Cells':>6s}")
print(f"  {'-'*56}")
for s in season_order:
    s_df = full_df[full_df["season"] == s]
    print(f"  {labels[s]:<30s} {len(s_df):>8,} {s_df['taxon_id'].nunique():>8,} {s_df['grid_cell'].nunique():>6,}")

# Winter₀ vs Winter₁ comparison
w0 = set(full_df[full_df["season"] == "winter_0"]["taxon_id"].unique())
w1 = set(full_df[full_df["season"] == "winter_1"]["taxon_id"].unique())
if w0 and w1:
    jac = len(w0 & w1) / len(w0 | w1)
    ovl = len(w0 & w1) / min(len(w0), len(w1))
    print(f"\n  Winter₀ vs Winter₁:")
    print(f"    Species W₀: {len(w0):,}  |  Species W₁: {len(w1):,}  |  Shared: {len(w0 & w1):,}")
    print(f"    Jaccard: {jac:.3f}  |  Overlap: {ovl:.3f}")



📅 Crawling Dec 2024  (2024-12-01 → 2024-12-31)
  Estimated observations: 21,930


  Dec 2024: 100%|██████████| 21930/21930 [10:29<00:00, 34.86 obs/s]


  ✅ 21,737 animal observations

📅 Crawling Jan 2026  (2026-01-01 → 2026-01-31)
  Estimated observations: 23,070


  Jan 2026: 100%|██████████| 23070/23070 [11:03<00:00, 34.79 obs/s]


  ✅ 22,982 animal observations

📅 Crawling Feb 2026  (2026-02-01 → 2026-02-28)
  Estimated observations: 36,056


  Feb 2026: 100%|██████████| 36056/36056 [16:45<00:00, 35.87 obs/s]


  ✅ 35,854 animal observations

💾 Supplementary saved: data\observations_supplementary.csv (80,573 rows)

🔄 Merging with existing observations...
  Original: 631,778 rows
  Supplementary: 80,573 rows

💾 Extended dataset saved: data\observations_extended.csv (681,813 rows)

📋 EXTENDED DATASET — 5-Snapshot Timeline
  Total observations:  681,813
  Unique species:      12,592
  Unique grid cells:   6,475

  Season                              Obs  Species  Cells
  --------------------------------------------------------
  Winter₀ (Dec24+Jan-Feb25)        46,090    2,655  3,121
  Spring  (Mar-May 2025)          181,780    7,357  5,215
  Summer  (Jun-Aug 2025)          291,297    8,863  5,639
  Autumn  (Sep-Nov 2025)          110,966    5,503  4,849
  Winter₁ (Dec25+Jan-Feb26)        51,680    2,682  3,336

  Winter₀ vs Winter₁:
    Species W₀: 2,655  |  Species W₁: 2,682  |  Shared: 1,650
    Jaccard: 0.448  |  Overlap: 0.621
